### 1. Intro
The tokenizer is the unsung hero of the LLM landscape. It’s the component that turns text into numbers or,
better yet, into their corresponding indices in its vocabulary, that is, their token IDs. The tokenizer’s
vocabulary is the universe of all tokens (words or sub-words) that it knows. If something isn’t in the vocabulary, it doesn’t exist as far as the tokenizer is concerned. In practice, this is rare, as subword tokenizers are designed to handle a wide variety of inputs. These tokenizers may split new (or even made-up words) into smaller components so that no word is left behind. For this reason, when tokenizing a sequence of words, we’re likely ending up with a longer sequence of tokens as a result—since some words may be represented by two or more tokens.

Most tokens are ordinary words, prefixes, or suffixes, but some tokens are truly special–because we made
them so. Special tokens serve as signposts to many things, such as:
- to inform the model that something is present merely for padding and has no actual meaning (the PAD token),
- to indicate the boundary between two independent sequences (the SEP token), and
- to signal the actual end of the text as a whole (the EOS token).

There are typically seven special tokens in total. These special tokens are added to the mix according to the
tokenizer’s configuration.

But we may also create additional special tokens of our own making to represent specific roles or structures
in our data. For example, tokens like <|user|> and <|assistant|> can define roles in a conversation, helping
the model understand context and assign responses appropriately.

### 2. Inputs to the token - Formatting and the Birth of Instruction Finetuning 


A base model like GPT-2, with their only job being next-token-prediction, given an initial prompt such as "The capital of Argentina is," the model would happily complete the sentence and keep on rambling: "Buenos Aires, located at…". However, getting the model to answer something was
cumbersome, as questions had to be reframed as statements with blanks at the end. That’s the motivation behind
instruction or chat models: give the model an instruction or ask it a question, and get a proper and limited (as
opposed to never-ending) reply from it

The one simple and easy way of finetuning an instruction model is to add a tiny bit of formatting to your prompt:
- One special token that lets the model know that the user is finished and it’s time for the model to take a turn
and "talk."
    - This special token is often referred to as generation prompt or response template.
    - During training, it indicates the starting point of the expected completion.
    - In inference time, it triggers the model’s response.
- One special token placed at the end of the model’s answer (in the training set) to signal the model that "it’s
enough, you can stop talking."
    - This special token is known as the EOS (end-of-sentence) token.
    - During training, it indicates the end of the expected completion.
    - In inference time, it halts the model’s generation.


Now, the prompt may be an actual question: "What is the capital of Argentina?" The desired answer is short and
to the point, "Buenos Aires," followed by an EOS token that indicates the answer should end right there.

For simple pairs of inputs and outputs, such as one-off interactions consisting of short questions and answers,
this very simple formatting template is already sufficient.

However, chat interactions, we’d better help it keep track of who said what and when. To accomplish this, we can improve our formatting by wrapping the data with some additional special tokens:
- the instruction template, preceding the user’s input–usually called the prompt, and
- the response template, preceding the assistant’s expected answer–referred to as the completion.


There are many templates available out there—each model uses a slightly different one:
- Some templates use the roles themselves as tokens (e.g. <|user|>, <|assistant|>, <|system|>), while
others use generic tokens followed by the role itself and a line-break (e.g., <|im_start|> user\n,
<|im_start|> assistant\n, <|im_start|> system\n)
- While many templates use a special ending token to wrap each message (e.g., <|im_end|>), others use the
EOS token instead (e.g., </s>, <|endoftext|>).
- Some templates may include that extra EOS token at the end of the whole thing, as shown in the previous
figure.
- This extra EOS token is especially important if the end of each message is indicated by any token other
than EOS

</br>

NOTE: If the model has been trained or fine-tuned using a specific template, you absolutely must
use that exact same template if you want to run inference on that model. If you’re fine
tuning, you could potentially use something different if absolutely necessary, but you’ll likely
be better off sticking with the template it’s already familiar with. Don’t reinvent the wheel.

### 2.1 Inspecting chat templates

In [100]:
import torch
from datasets import load_dataset, Dataset
from peft import prepare_model_for_kbit_training, get_peft_model, LoraConfig
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig, DataCollatorForLanguageModeling, DataCollatorWithPadding, DataCollatorWithFlattening, BitsAndBytesConfig, DataCollator
#from trl import setup_chat_format
from trl.data_utils import pack_dataset
from trl.extras.dataset_formatting import FORMAT_MAPPING

In [8]:
# picking an instruct model

repo_id = "microsoft/phi-3-mini-4k-instruct"
tokenizer_phi = AutoTokenizer.from_pretrained(repo_id)
print(tokenizer_phi.chat_template)

{% for message in messages %}{% if message['role'] == 'system' %}{{'<|system|>
' + message['content'] + '<|end|>
'}}{% elif message['role'] == 'user' %}{{'<|user|>
' + message['content'] + '<|end|>
'}}{% elif message['role'] == 'assistant' %}{{'<|assistant|>
' + message['content'] + '<|end|>
'}}{% endif %}{% endfor %}{% if add_generation_prompt %}{{ '<|assistant|>
' }}{% else %}{{ eos_token }}{% endif %}


In [9]:
messages = [
    {'role': 'system', 'content': 'You are a helpful AI assistant.'},
    {'role': 'user',  'content': 'What is the capital of Argentina?'},
    {'role': 'assistant', 'content': 'Buenos Aires.'}
]
formatted = tokenizer_phi.apply_chat_template(
    conversation=messages, tokenize=False, add_generation_prompt=False
)
print(formatted)

<|system|>
You are a helpful AI assistant.<|end|>
<|user|>
What is the capital of Argentina?<|end|>
<|assistant|>
Buenos Aires.<|end|>
<|endoftext|>


In [10]:
inference_input = tokenizer_phi.apply_chat_template(
    conversation=messages[:-1], tokenize=False, add_generation_prompt=True
)
print(inference_input)

<|system|>
You are a helpful AI assistant.<|end|>
<|user|>
What is the capital of Argentina?<|end|>
<|assistant|>



In [11]:
conversation_ds = Dataset.from_list([{'messages': messages}])
conversation_ds.features

{'messages': List({'content': Value('string'), 'role': Value('string')})}

In [12]:
FORMAT_MAPPING['chatml'] == conversation_ds.features['messages']

True

In [111]:
def byofd_formatting_func(examples):
    output = []
    for i in range(len(examples["messages"])):
        output.append(
            tokenizer_phi.apply_chat_template(
                conversation = examples["messages"][i], tokenize=False, padding = False, 
                truncation = False, 
                add_generation_prompt = False)
                )
    return {"text": output}

In [106]:
help(tokenizer_phi.apply_chat_template)

Help on method apply_chat_template in module transformers.tokenization_utils_base:

apply_chat_template(conversation: 'list[dict[str, str]] | list[list[dict[str, str]]]', tools: 'list[dict | Callable] | None' = None, documents: 'list[dict[str, str]] | None' = None, chat_template: 'str | None' = None, add_generation_prompt: 'bool' = False, continue_final_message: 'bool' = False, tokenize: 'bool' = True, padding: 'bool | str | PaddingStrategy' = False, truncation: 'bool' = False, max_length: 'int | None' = None, return_tensors: 'str | TensorType | None' = None, return_dict: 'bool' = True, return_assistant_tokens_mask: 'bool' = False, tokenizer_kwargs: 'dict[str, Any] | None' = None, **kwargs) -> 'str | list[int] | list[str] | list[list[int]] | BatchEncoding' method of transformers.tokenization_utils_tokenizers.TokenizersBackend instance
    Converts a list of dictionaries with `"role"` and `"content"` keys to a list of token
    ids. This method is intended for use with chat models, and 

In [22]:
messages

[{'role': 'system', 'content': 'You are a helpful AI assistant.'},
 {'role': 'user', 'content': 'What is the capital of Argentina?'},
 {'role': 'assistant', 'content': 'Buenos Aires.'}]

In [112]:
batched_messages = [
    [{'role': 'user',
      'content': 'What is the capital of Argentina?'},        
     {'role': 'assistant',
      'content': 'Buenos Aires.'}],                           
    [{'role': 'user',
      'content': 'What is the capital of the United States?'},
     {'role': 'assistant',
        'content': 'Washington, D.C.'}]
]
ds_msg = Dataset.from_dict({'messages': batched_messages})
ds_msg

Dataset({
    features: ['messages'],
    num_rows: 2
})

In [113]:
ds_msg[0]

{'messages': [{'content': 'What is the capital of Argentina?', 'role': 'user'},
  {'content': 'Buenos Aires.', 'role': 'assistant'}]}

In [114]:
formatted_ds = ds_msg.map(byofd_formatting_func, batched=True)
formatted_ds['text']

Map: 100%|██████████| 2/2 [00:00<00:00, 146.86 examples/s]


Column(['<|user|>\nWhat is the capital of Argentina?<|end|>\n<|assistant|>\nBuenos Aires.<|end|>\n<|endoftext|>', '<|user|>\nWhat is the capital of the United States?<|end|>\n<|assistant|>\nWashington, D.C.<|end|>\n<|endoftext|>'])

### 3. Feed into Tokenizer

In [33]:
repo_id = "microsoft/phi-3-mini-4k-instruct"
tokenizer_phi = AutoTokenizer.from_pretrained(repo_id)
config_phi = AutoConfig.from_pretrained(repo_id, trust_remote_code=True)

c:\Users\Shruti\anaconda3\envs\slm_env\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Shruti\.cache\huggingface\hub\models--microsoft--phi-3-mini-4k-instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
A new version of the following files was downloaded from https://huggingface.co/microsoft/phi-3-

In [ ]:
# # Masks are used to indicate which tokens should be kept (1) and included in the computation
# # or ignored (0). Padding tokens, being meaningless, are the most obvious and straightforward
# # targets to ignore. As we’ll see later, padding tokens will have zero values in their
# # corresponding masks

# The secondary duties of the tokenizer include padding, truncating, and adding other special tokens to its
# output. Finally, tokenizers have also been recruited to define and apply chat templates to their inputs

tokenizer_phi("Let's tokenize this sentence!")

{'input_ids': [2803, 29915, 29879, 5993, 675, 445, 10541, 29991], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}

The tokenizer’s vocabulary is the exhaustive list of every possible token a model can handle. Each token
corresponds to a unique token ID, which in turn serves as an index to a large lookup table of embeddings. It
should follow that the length of the embedding layer matches the size of the vocabulary, right?
Not necessarily. Sure, the embedding layer **cannot** be shorter than the vocabulary; otherwise, we’d get index
errors all the time. But it can, and it often is, longer than the vocabulary. As it turns out, it’s more memory-efficient to keep the dimensions of the embedding layer as a multiple of a power of two. So most models nowadays have embedding layers longer than what is actually required by the
corresponding vocabulary.

In [35]:
len(tokenizer_phi), config_phi.vocab_size

(32011, 32064)

In [36]:
sorted(tokenizer_phi.vocab.items(), key=lambda t: -t[1])[:11]

[('<|user|>', 32010),
 ('<|placeholder6|>', 32009),
 ('<|placeholder5|>', 32008),
 ('<|end|>', 32007),
 ('<|system|>', 32006),
 ('<|placeholder4|>', 32005),
 ('<|placeholder3|>', 32004),
 ('<|placeholder2|>', 32003),
 ('<|placeholder1|>', 32002),
 ('<|assistant|>', 32001),
 ('<|endoftext|>', 32000)]

In [37]:
tokenizer_phi.eos_token, tokenizer_phi.eos_token_id

('<|endoftext|>', 32000)

### 3.1 Common Special Tokens

Typically, there are seven special tokens. Two of them are especially important to us:
- EOS: The end-of-sentence token, marking the end of a full sequence—it signals to the model that it’s time to
stop generating tokens.
- PAD: The padding token. Its role is simple—padding or stuffing sequences to ensure matching lengths—but
its potential impact is significant, so we’ll dedicate an entire section to it.
The other five tokens are:
- BOS: The beginning-of-sentence token, marking the start of an input sequence.
- UNK: The unknown token, is used to replace tokens not found in the vocabulary (though it’s rarely used in
sub-word tokenizers).
- CLS: The classifier token, acting as a "summary" of the sentence and commonly used in classification tasks,
as its name suggests.
- SEP: The separation token, indicating where a sentence ends within a sequence of tokens.
- MASK: The mask token, is used to make a specific token invisible to the model (e.g., in masked language
modeling tasks).

In [38]:
tokenizer_phi.all_special_tokens # eos for phi-3 also acts as the padding token

['<s>', '<|endoftext|>', '<unk>']

In [39]:
tokenizer_phi.special_tokens_map

{'bos_token': '<s>',
 'eos_token': '<|endoftext|>',
 'unk_token': '<unk>',
 'pad_token': '<|endoftext|>'}

In [40]:
# we can sometimes add special tokens because the embedding layer is bigger than the vocab size, 
# in that case the new tokens will be initialized with random embeddings and the model will learn to 
# use them during training.
tokenizer_phi.add_special_tokens({
'cls_token': '<cls>', 'sep_token': '<sep>', 'mask_token': '<mask>'
})
tokenizer_phi.special_tokens_map

{'bos_token': '<s>',
 'eos_token': '<|endoftext|>',
 'unk_token': '<unk>',
 'sep_token': '<sep>',
 'pad_token': '<|endoftext|>',
 'cls_token': '<cls>',
 'mask_token': '<mask>'}

### 3.1.1 Padding and its token

Padding is a solution to a collation problem

IMP: Using EOS token as padding is not encouraged but a very common practice in fine-tuning. In the SFTTrainer if the padding token is not defined, it is by default set to EOS token. This is still common because actual padding isn’t happening nowadays: the sequences are **packed** instead. In some sense, the padding token feels like an afterthought.

If padding token is not defined, it is better to use UNK as the padding token

In [41]:
tokenizer_phi.pad_token = tokenizer_phi.unk_token
tokenizer_phi.pad_token_id = tokenizer_phi.unk_token_id
tokenizer_phi.special_tokens_map

{'bos_token': '<s>',
 'eos_token': '<|endoftext|>',
 'unk_token': '<unk>',
 'sep_token': '<sep>',
 'pad_token': '<unk>',
 'cls_token': '<cls>',
 'mask_token': '<mask>'}

For generative language models, the right way of padding is not right- but left-padding. The model doesn't need to learn to generate padding tokens.

In [42]:
tokenizer_phi.pad_token, tokenizer_phi.padding_side

('<unk>', 'left')

### 3.2 Data Collators

The data collator is responsible for stitching together multiple data points into a mini-batch. It’s usually
invisible to us. Every time we use PyTorch’s DataLoader, we're relying on its default data collator without even
realizing it. The first action we need to take to run a collator successfully, we need to make sure that the input tensors have the same size.


The culprit is, more often than not, sequences of varied lengths. We cannot stitch together tensors of different
sizes and, as the default collator tries to perform its job, an exception is raised. At this point, we become aware
of its existence and we scramble to replace it using the collate_fn argument of our data loader.

Below we first make our data compatible with the conversational format.

In [115]:
def make_messages_format_dataset(examples):
    """Converts prompt/completion pairs to conversational messages format."""
    if isinstance(examples["prompt"], list):
        output_texts = []
        for i in range(len(examples["prompt"])):
            converted_sample = [
                {"role": "user", "content": examples["prompt"][i]},
                {"role": "assistant", "content": examples["completion"][i]},
            ]
            output_texts.append(converted_sample)
        return {'messages': output_texts}
    else:
        converted_sample = [
            {"role": "user", "content": examples["prompt"]},
            {"role": "assistant", "content": examples["completion"]},
        ]
        return {'messages': converted_sample}

In [116]:
dataset = load_dataset("dvgodoy/yoda_sentences", split="train")
dataset = dataset.rename_column("sentence", "prompt")
dataset = dataset.rename_column("translation_extra", "completion")
# converts prompt/completion pairs to conversational messages
dataset = dataset.map(make_messages_format_dataset, batched=True)
dataset = dataset.remove_columns(["prompt", "completion", "translation"])
len(dataset), dataset[0]

(720,
 {'messages': [{'content': 'The birch canoe slid on the smooth planks.',
    'role': 'user'},
   {'content': 'On the smooth planks, the birch canoe slid. Yes, hrrrm.',
    'role': 'assistant'}]})

In [117]:
dataset

Dataset({
    features: ['messages'],
    num_rows: 720
})

In [118]:
dataset = dataset.map(byofd_formatting_func,
                      batched=True, batch_size=32)
sequences = dataset['text']
print(sequences[0])

Map:   0%|          | 0/720 [00:00<?, ? examples/s]

Map: 100%|██████████| 720/720 [00:00<00:00, 5955.29 examples/s]

<|user|>
The birch canoe slid on the smooth planks.<|end|>
<|assistant|>
On the smooth planks, the birch canoe slid. Yes, hrrrm.<|end|>
<|endoftext|>


In [119]:
len(sequences)

720

In [143]:
# to tokenize our dataset and keep only the resulting token IDs
tokenized_dataset = dataset.map(lambda row: tokenizer_phi(row['text']))
tokenized_dataset = tokenized_dataset.select_columns(['input_ids'])

In [144]:
tokenized_dataset

Dataset({
    features: ['input_ids'],
    num_rows: 720
})

#### Default Data Collator

In [138]:
from transformers import DefaultDataCollator

default_collator = DefaultDataCollator(tokenizer_phi,)
default_dloader = DataLoader(tokenized_dataset, batch_size=2, collate_fn=default_collator)
batch = next(iter(default_dloader))
batch

In [136]:
tokenized_dataset[0].keys()

dict_keys(['input_ids', 'attention_mask'])

We see that the batch is None, this is unexpected. On further research, we find that there seems to be a rule of thumb for using collators:

- DefaultDataCollator  
This collator is meant for supervised training tasks. It assumes your dataset has not only input_ids and attention_mask, but also a labels field (or label). Its job is to batch and pad all of those together. If it doesn’t find labels, it often returns None because it thinks there’s nothing to collate for training.

- DataCollatorWithPadding  
This one is more general-purpose. It simply takes whatever fields the tokenizer produced (input_ids, attention_mask, maybe token_type_ids) and pads them dynamically to the longest sequence in the batch. It doesn’t require labels, so it works fine for inference or unsupervised setups.


Therefore,

- Use DataCollatorWithPadding if you only want to batch inputs.
- Use DefaultDataCollator if you also have labels for supervised training.

#### Data Collator with Padding

In [ ]:
from transformers import DataCollatorWithPadding

pad_collator = DataCollatorWithPadding(tokenizer=tokenizer_phi)
pad_dloader = DataLoader(tokenized_dataset, batch_size=2, collate_fn=pad_collator)
pad_batch = next(iter(pad_dloader))
pad_batch # see the second sequence has been padded to the same length as the first one, and an attention mask has been added to indicate which tokens are relevant and which are not.

{'input_ids': tensor([[32010,   450, 29773,   305,   508,  7297,  2243,   333,   373,   278,
         10597,   715,  1331, 29889, 32007, 32001,  1551,   278, 10597,   715,
          1331, 29892,   278, 29773,   305,   508,  7297,  2243,   333, 29889,
          3869, 29892,   298, 21478,  1758, 29889, 32007, 32000],
        [    0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
         32010,  8467,   434,   278,  9869,   304,   278,  6501,  7254,  3239,
         29889, 32007, 32001,  8467,   434,   278,  9869,   304,   278,  6501,
          7254,  3239, 29892,   366,  1818, 29889, 32007, 32000]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [146]:
tokenizer_phi.pad_token_id

0

How do we see attention mask? We didn't pass it to the collator or dataloader
- DataCollatorWithPadding (and DefaultDataCollator) use the tokenizer’s pad method.
- The tokenizer’s pad method always produces attention_mask alongside padded input_ids.


That’s why you get attention_mask “for free” in your batches.

BUT! We don't see any labels, what's the model going to train on?

#### Data Collator for Language Modeling

This collator was built, as its name suggests, for language modeling or, in other words, for self-supervised
tasks. You know, those tasks where labels are exactly the same as the inputs (**except for shifting**, more on that
soon). This is the default collator used by the SFTTrainer class. So, if you’re padding
rather than packing your dataset, this is
what’s happening under the hood.

In [147]:
from transformers import DataCollatorForLanguageModeling

lm_collator = DataCollatorForLanguageModeling(tokenizer_phi, mlm=False)
lm_dloader = DataLoader(tokenized_dataset, batch_size=2, collate_fn=lm_collator)
lm_batch = next(iter(lm_dloader))
lm_batch

{'input_ids': tensor([[32010,   450, 29773,   305,   508,  7297,  2243,   333,   373,   278,
         10597,   715,  1331, 29889, 32007, 32001,  1551,   278, 10597,   715,
          1331, 29892,   278, 29773,   305,   508,  7297,  2243,   333, 29889,
          3869, 29892,   298, 21478,  1758, 29889, 32007, 32000],
        [    0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
         32010,  8467,   434,   278,  9869,   304,   278,  6501,  7254,  3239,
         29889, 32007, 32001,  8467,   434,   278,  9869,   304,   278,  6501,
          7254,  3239, 29892,   366,  1818, 29889, 32007, 32000]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]), 'labels': tensor([[32010,   450, 29773,   305,   508,  7297,  2243,   333,   373,   278,
   

Not only do we have padded sequences, but we also have them replicated as labels. Notice that
padded tokens are indicated by -100 when they are part of the labels. If you try to use the tokenizer to decode
these labels, you’ll get an error because -100 is not a valid token. However; it is gracefully
handled during the training process.

Also, we have set `mlm` param to false. It indicates masked language model based training where tokens would be randomly removed/masked from the input to be the labels. It is relevant to encoder-decoder models. Since we’re dealing with generative models, decoder-only, we don’t really care about masking tokens, so we must keep this argument False

##### IMP: Even though we got the labels ready, if we look carefully, the labels after shifting left one step, the prompt still remains a part of the label. Now if the model being trained on was a base model, it is good to keep the prompt in the label too because the model has really never seen an instruction or question coming to it. It's good. But if not, then training on prompt tokens would be a waste of resources. A smarter model already knows how instructions and questions are framed. If the model is already pretrained and "fluent" in English, and the prompts are ordinary sentences, we’d be spending time and resources teaching the model something it already knows pretty well. There’s nothing new to learn from the prompt itself. It’s the answer–the completion–that contains the useful information we’re trying to teach the model, isn’t it? Perhaps we’re trying to teach the model to answer in a different tone. We'd like it to respond using a different dialect or follow some specific structure. These are use cases that could benefit from fine-tuning on **"completions only."** This is where we use **DataCollatorForCompletionOnlyLM**.

###### NOTE: The DataCollatorForCompletionOnlyLM was removed in trl version 0.20. This collator masked the user prompt to train only on the LM’s completion, excluding prompt tokens from the loss computation by using the response template to detect the start of the completion. 
In newer versions, this logic is built-in: tokens are automatically ignored during loss computation if completion_only_loss or assistant_only_loss are set in the SFTConfig object.

This collator or the built-in logic in recent versions uses the response template to identify two parts of the input:
- the prompt (whatever comes before the response template)
- the completion (whatever comes after the response template)

Tokens belonging to the prompt will not make it to the labels, being replaced by -100 (as if they were padding
tokens in the input). It makes sense: padding tokens are ignored as labels, and if we’re training our model on
completions only, the prompt should be ignored as well.

### 3.3 Data Packing

It's visible that to be able to successfully process batches data collators pad the sequences from the left but there's no actual information that they hold and they end up wasting resources too. The idea of packing is quite straightforward: concatenate all sequences one after the other (including a separator), split them into equal-sized chunks (the chosen sequence length), and shuffle the resulting chunks. 

In [148]:
### 3.3 Data Packing